# Tutorial: Complex Tensor Networks by Hand

Audience:
- Readers of this repository who want examples that are more visual and structurally richer than a single toy contraction.

Prerequisites:
- Basic NumPy arrays.
- A rough idea of what a tensor and a contraction are.

Learning goals:
- Build nontrivial tensor networks manually with `tn.Node` and `^`.
- Render the same network in both `2D` and `3D` with `tensor_network_viz`.
- Contract three different topologies in a controlled way.
- Compare what changes when the network is layered, loopy, or hierarchical.

The tensors in this notebook are mostly normalized random arrays. The point is not domain semantics; the point is to learn how topology, visualization, and contraction interact in `tensornetwork`.


## Outline

1. Shared setup and plotting helpers.
2. A layered feed-forward style network.
3. A loopy lattice patch with one open probe leg.
4. A hierarchical tree network.
5. A closing comparison of the three topologies.


## Shared Setup

The helper cell below does four things:

- creates deterministic normalized tensors from integer seeds,
- recovers the connected component of a network with `tn.reachable(...)`,
- renders each example in `2D` and `3D`,
- prints compact shape summaries after contraction.

The plotting helper is careful about headless verification: if the backend is non-interactive, it closes the figures instead of calling `plt.show()` and producing warnings.


In [ ]:
from __future__ import annotations

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import tensornetwork as tn

from tensor_network_viz import PlotConfig, show_tensor_network

np.set_printoptions(precision=4, suppress=True)


def normalized_tensor(shape, seed: int):
    """Create a deterministic random tensor with unit Frobenius norm."""
    data = np.random.default_rng(seed).normal(size=shape)
    norm = np.linalg.norm(data.reshape(-1))
    return data / norm if norm else data


def reachable_nodes(seed_node: tn.Node):
    """Return the connected component containing the chosen seed node."""
    return sorted(tn.reachable(seed_node), key=lambda node: node.name)


def _show_or_close(fig):
    backend = matplotlib.get_backend().lower()
    if 'agg' in backend:
        plt.close(fig)
    else:
        plt.show()


def render_2d_3d(
    seed_node: tn.Node,
    title: str,
    *,
    figsize_2d=(8, 5),
    figsize_3d=(8, 6),
    layout_iterations=300,
):
    """Render the same connected component in two different views."""
    nodes = reachable_nodes(seed_node)

    fig2d, ax2d = show_tensor_network(
        nodes,
        engine='tensornetwork',
        view='2d',
        config=PlotConfig(figsize=figsize_2d, layout_iterations=layout_iterations),
        show=False,
    )
    ax2d.set_title(f'{title} (2D)')
    _show_or_close(fig2d)

    fig3d, ax3d = show_tensor_network(
        nodes,
        engine='tensornetwork',
        view='3d',
        config=PlotConfig(figsize=figsize_3d, layout_iterations=layout_iterations),
        show=False,
    )
    ax3d.set_title(f'{title} (3D)')
    _show_or_close(fig3d)

    return nodes


def shape_report(label: str, node_or_tensor):
    tensor = node_or_tensor.tensor if hasattr(node_or_tensor, 'tensor') else node_or_tensor
    print(f'{label}: shape={tensor.shape}')


## Family 1 - Layered Feed-Forward Network

This topology is acyclic: information moves from inputs to hidden tensors to a readout tensor, with no loops. It is a good first complex example because the contraction order is easy to reason about.

Structurally, the network has:

- three input vectors,
- two intermediate tensors,
- one readout tensor with a small output leg called `class`.

The main idea is that layered networks are visually richer than a single matrix multiplication, but still easy to contract in stages.


In [ ]:
def build_layered_network():
    x0 = tn.Node(normalized_tensor((3,), 10), name='x0', axis_names=['f0'])
    x1 = tn.Node(normalized_tensor((3,), 11), name='x1', axis_names=['f1'])
    x2 = tn.Node(normalized_tensor((3,), 12), name='x2', axis_names=['f2'])

    h0 = tn.Node(normalized_tensor((3, 3, 4), 20), name='h0', axis_names=['f0', 'f1', 'h0'])
    h1 = tn.Node(normalized_tensor((3, 4, 4), 21), name='h1', axis_names=['f2', 'h0_in', 'h1'])
    readout = tn.Node(normalized_tensor((4, 2), 22), name='readout', axis_names=['h1_in', 'class'])

    x0['f0'] ^ h0['f0']
    x1['f1'] ^ h0['f1']
    h0['h0'] ^ h1['h0_in']
    x2['f2'] ^ h1['f2']
    h1['h1'] ^ readout['h1_in']

    return {
        'x0': x0,
        'x1': x1,
        'x2': x2,
        'h0': h0,
        'h1': h1,
        'readout': readout,
    }


layered = build_layered_network()
layered_nodes = render_2d_3d(layered['x0'], 'Layered feed-forward network')
print('Layered node count:', len(layered_nodes))
print('Dangling edges:', sum(len(node.get_all_dangling()) for node in layered_nodes))


In [ ]:
layered = build_layered_network()
x0 = layered['x0']
x1 = layered['x1']
x2 = layered['x2']
h0 = layered['h0']
h1 = layered['h1']
readout = layered['readout']

layer01 = tn.contract_between(
    x0,
    h0,
    name='x0_h0',
    output_edge_order=[x1['f1'], h0['h0']],
    axis_names=['f1', 'h0'],
)
layer02 = tn.contract_between(
    x1,
    layer01,
    name='hidden0',
    output_edge_order=[layer01['h0']],
    axis_names=['h0'],
)
layer03 = tn.contract_between(
    layer02,
    h1,
    name='hidden1',
    output_edge_order=[x2['f2'], h1['h1']],
    axis_names=['f2', 'h1'],
)
layer04 = tn.contract_between(
    x2,
    layer03,
    name='pre_readout',
    output_edge_order=[layer03['h1']],
    axis_names=['h1'],
)
layer05 = tn.contract_between(
    layer04,
    readout,
    name='output',
    output_edge_order=[readout['class']],
    axis_names=['class'],
)

shape_report('Layered output', layer05)
print(layer05.tensor)


## Family 2 - Loopy Lattice Patch

This example is closer to a small PEPS-like patch. Unlike the layered network, it contains cycles: every square in the patch introduces a loop in the connectivity graph.

To make the final contraction readable, most boundary legs are closed with rank-1 probe tensors. One boundary leg, the top edge of the middle site, is left open so that the full patch contracts to a small vector instead of to a giant high-rank tensor.

This is the visually richest topology in the notebook and the one where `2D` and `3D` views help the most.


In [ ]:
def build_lattice_patch():
    sites = {}
    for row in range(2):
        for col in range(3):
            sites[row, col] = tn.Node(
                normalized_tensor((2, 2, 2, 2), 100 + 10 * row + col),
                name=f'A{row}{col}',
                axis_names=['up', 'right', 'down', 'left'],
            )

    for row in range(2):
        for col in range(3):
            if col < 2:
                sites[row, col]['right'] ^ sites[row, col + 1]['left']
            if row < 1:
                sites[row, col]['down'] ^ sites[row + 1, col]['up']

    boundary = []

    for col in range(3):
        if col != 1:
            probe = tn.Node(
                normalized_tensor((2,), 200 + col),
                name=f'top{col}',
                axis_names=[f'top{col}'],
            )
            probe[f'top{col}'] ^ sites[0, col]['up']
            boundary.append(probe)

    for col in range(3):
        probe = tn.Node(
            normalized_tensor((2,), 210 + col),
            name=f'bottom{col}',
            axis_names=[f'bottom{col}'],
        )
        probe[f'bottom{col}'] ^ sites[1, col]['down']
        boundary.append(probe)

    for row in range(2):
        probe = tn.Node(
            normalized_tensor((2,), 220 + row),
            name=f'left{row}',
            axis_names=[f'left{row}'],
        )
        probe[f'left{row}'] ^ sites[row, 0]['left']
        boundary.append(probe)

    for row in range(2):
        probe = tn.Node(
            normalized_tensor((2,), 230 + row),
            name=f'right{row}',
            axis_names=[f'right{row}'],
        )
        probe[f'right{row}'] ^ sites[row, 2]['right']
        boundary.append(probe)

    all_nodes = list(sites.values()) + boundary
    return sites, boundary, all_nodes


lattice_sites, lattice_boundary, lattice_nodes = build_lattice_patch()
render_2d_3d(
    lattice_sites[0, 0],
    'Loopy lattice patch',
    figsize_2d=(9, 5),
    figsize_3d=(9, 7),
    layout_iterations=350,
)
print('Lattice node count:', len(lattice_nodes))
print('Remaining open leg:', lattice_sites[0, 1]['up'])


In [ ]:
lattice_sites, lattice_boundary, lattice_nodes = build_lattice_patch()

lattice_result = tn.contractors.greedy(
    lattice_nodes,
    output_edge_order=[lattice_sites[0, 1]['up']],
)

shape_report('Lattice result', lattice_result)
print(lattice_result.tensor)


## Family 3 - Hierarchical Tree Network

Trees have depth, but no loops. That makes them a good contrast with the lattice patch.

This example starts with eight leaf vectors. They are merged upward by two layers of rank-3 tensors and then reduced at the root. The contraction is naturally bottom-up: first combine leaves into small summaries, then combine those summaries again, and only at the end hit the root.

In other words, the topology is not sequential like the layered network and not cyclic like the lattice; it is reduction-oriented.


In [ ]:
def build_tree_network():
    leaves = [
        tn.Node(normalized_tensor((3,), 300 + i), name=f'leaf{i}', axis_names=[f'x{i}'])
        for i in range(8)
    ]

    merge_a = [
        tn.Node(
            normalized_tensor((3, 3, 4), 400 + i),
            name=f'merge_a{i}',
            axis_names=[f'l{i}', f'r{i}', f'p{i}'],
        )
        for i in range(4)
    ]

    merge_b = [
        tn.Node(
            normalized_tensor((4, 4, 5), 500 + i),
            name=f'merge_b{i}',
            axis_names=[f'l2_{i}', f'r2_{i}', f'p2_{i}'],
        )
        for i in range(2)
    ]

    root = tn.Node(
        normalized_tensor((5, 5), 600),
        name='root',
        axis_names=['left_root', 'right_root'],
    )

    for i in range(4):
        leaves[2 * i][f'x{2 * i}'] ^ merge_a[i][f'l{i}']
        leaves[2 * i + 1][f'x{2 * i + 1}'] ^ merge_a[i][f'r{i}']

    for i in range(2):
        merge_a[2 * i][f'p{2 * i}'] ^ merge_b[i][f'l2_{i}']
        merge_a[2 * i + 1][f'p{2 * i + 1}'] ^ merge_b[i][f'r2_{i}']

    merge_b[0]['p2_0'] ^ root['left_root']
    merge_b[1]['p2_1'] ^ root['right_root']

    return leaves, merge_a, merge_b, root


tree_leaves, tree_merge_a, tree_merge_b, tree_root = build_tree_network()
render_2d_3d(
    tree_leaves[0],
    'Hierarchical tree network',
    figsize_2d=(9, 5),
    figsize_3d=(9, 7),
    layout_iterations=350,
)
print('Tree node count:', len(tree_leaves) + len(tree_merge_a) + len(tree_merge_b) + 1)
print('Dangling edges at the leaves:', sum(len(node.get_all_dangling()) for node in tree_leaves))


In [ ]:
tree_leaves, tree_merge_a, tree_merge_b, tree_root = build_tree_network()

level1 = []
for i in range(4):
    left = tn.contract_between(tree_leaves[2 * i], tree_merge_a[i], name=f'leaf_merge_{i}_left')
    merged = tn.contract_between(tree_leaves[2 * i + 1], left, name=f'level1_{i}')
    level1.append(merged)

level2 = []
for i in range(2):
    left = tn.contract_between(level1[2 * i], tree_merge_b[i], name=f'level2_left_{i}')
    merged = tn.contract_between(level1[2 * i + 1], left, name=f'level2_{i}')
    level2.append(merged)

top = tn.contract_between(level2[0], tree_root, name='top')
tree_result = tn.contract_between(level2[1], top, name='tree_result')

shape_report('Tree result', tree_result)
print(tree_result.tensor)


## Topology Comparison

- **Layered feed-forward network:** acyclic, staged, and easy to contract in the same order in which it is drawn.
- **Loopy lattice patch:** locally regular, globally more tangled, and much more visually interesting because cycles create spatial structure.
- **Hierarchical tree network:** loop-free like the layered case, but not sequential; the natural contraction order is bottom-up.

The important point is that `tensornetwork` is not only for circuit diagrams. Once you are comfortable with `tn.Node`, named axes, and explicit `^` connections, you can model many different graph topologies and choose contraction paths that fit the structure in front of you.
